In [10]:
import pandas as pd
import numpy as np

In [24]:
train_df = pd.read_csv('antimicrobial_train_80pct.csv')
test_df = pd.read_csv('antimicrobial_test_X_20pct.csv')

display(train_df.head(3))
display(test_df.head(3))

,activity_id,molecule_chembl_id,pref_name,compound_name,canonical_smiles,standard_type,standard_units,assay_chembl_id,rdkit_MaxAbsEStateIndex,rdkit_MaxEStateIndex,...,rdkit_fr_sulfonamd,rdkit_fr_sulfone,rdkit_fr_term_acetylene,rdkit_fr_tetrazole,rdkit_fr_thiazole,rdkit_fr_thiocyan,rdkit_fr_thiophene,rdkit_fr_unbrch_alkane,rdkit_fr_urea,target_log10
0,946920,CHEMBL72286,NaN,NaN,O=C(c1ccccc1O)N1CCc2c(n(Cc3ccc(O)cc3)c3ccccc23)C1,MIC,ug.mL-1,CHEMBL677573,13.081525,13.081525,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.903090
1,379615,CHEMBL134905,NaN,NaN,CC(=O)NC[C@H]1CN(c2ccc3c(c2)CCCCN3C=O)C(=O)O1,MIC,ug.mL-1,CHEMBL677421,12.110715,12.110715,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.301030
2,16571259,CHEMBL3798190,NaN,NaN,CCn1cc(C(=O)O)c(=O)c2cc(F)c(N3CCNCC3)cc21.O=[N...,MIC,ug.mL-1,CHEMBL3801691,14.494909,14.494909,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.892095


,activity_id,molecule_chembl_id,pref_name,compound_name,canonical_smiles,standard_type,standard_units,assay_chembl_id,rdkit_MaxAbsEStateIndex,rdkit_MaxEStateIndex,...,rdkit_fr_sulfide,rdkit_fr_sulfonamd,rdkit_fr_sulfone,rdkit_fr_term_acetylene,rdkit_fr_tetrazole,rdkit_fr_thiazole,rdkit_fr_thiocyan,rdkit_fr_thiophene,rdkit_fr_unbrch_alkane,rdkit_fr_urea
0,849731,CHEMBL121649,NaN,NaN,O=C(Cc1ccccc1)N[C@@H]1C(=O)N2C(C(=O)O)=C(CCO)C...,MIC,ug.mL-1,CHEMBL673564,12.247022,12.247022,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,287078,CHEMBL68208,NaN,NaN,C[C@@H](O)[C@H]1C(=O)N2C(C(=O)[O-])=C(CSC(=S)N...,MIC,ug.mL-1,CHEMBL676864,12.092750,12.092750,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,863129,CHEMBL134718,NaN,NaN,O=C(NC[C@H]1CN(c2ccc(N3CCS(=O)(=O)CC3)c(F)c2)C...,MIC,ug.mL-1,CHEMBL679456,14.677104,14.677104,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
print(f"Train Shape: {train_df.shape}")
print(f"Test Shape:  {test_df.shape}")

Train Shape: (8475, 226)
Test Shape:  (2119, 225)


In [26]:
target_col = "target_log10"
y_train = train_df[target_col].copy()

In [27]:
metadata_cols = [
    "activity_id",
    "molecule_chembl_id",
    "pref_name",
    "compound_name",
    "canonical_smiles",
    "standard_type",
    "standard_units",
    "assay_chembl_id",
    target_col,  # Only present in train
]

feature_cols = [
    c
    for c in train_df.columns
    if c not in metadata_cols and c in test_df.columns
]

print(f"Total predictive features: {len(feature_cols)}")

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

Total predictive features: 217


In [28]:
# Detect adn handle NAs and infinite values

# convert infinite values to nan
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

pseudo_na_values = [-999, -9999, 999, 9999, 99999]

for val in pseudo_na_values:
    train_count = (X_train == val).sum().sum()
    if train_count > 0:
        print(f"Found {train_count} occurrences of pseudo-NA value: {val}")
        X_train = X_train.replace(val, np.nan)
        X_test = X_test.replace(val, np.nan)

# check true missing value percentages
missing_train = X_train.isnull().mean()
missing_cols = missing_train[missing_train > 0]
print("\nColumns with missing values in Train:")
print(missing_cols)


Columns with missing values in Train:
rdkit_MaxAbsEStateIndex    0.019823
rdkit_MaxEStateIndex       0.021121
rdkit_MinAbsEStateIndex    0.022773
rdkit_MinEStateIndex       0.017345
rdkit_qed                  0.021239
                             ...   
rdkit_fr_thiazole          0.020059
rdkit_fr_thiocyan          0.022183
rdkit_fr_thiophene         0.021947
rdkit_fr_unbrch_alkane     0.021593
rdkit_fr_urea              0.020059
Length: 217, dtype: float64


In [30]:
float32_max = np.finfo(np.float32).max
float32_min = np.finfo(np.float32).min

X_train = X_train.map(
    lambda x: np.nan
    if pd.notnull(x) and (x > float32_max or x < float32_min)
    else x
)
X_test = X_test.map(
    lambda x: np.nan
    if pd.notnull(x) and (x > float32_max or x < float32_min)
    else x
)

/tmp/ipykernel_1214/1771022037.py:6: RuntimeWarning: overflow encountered in cast
  if pd.notnull(x) and (x > float32_max or x < float32_min)
/tmp/ipykernel_1214/1771022037.py:11: RuntimeWarning: overflow encountered in cast
  if pd.notnull(x) and (x > float32_max or x < float32_min)


In [31]:
# identify constant columns (where max == min or std == 0)
constant_cols = [col for col in X_train.columns if X_train[col].nunique() <= 1]

print(
    f"Constant cols count {len(constant_cols)} constant features: {constant_cols}"
)

Constant cols count 11 constant features: ['rdkit_NumRadicalElectrons', 'rdkit_SMR_VSA8', 'rdkit_SlogP_VSA9', 'rdkit_fr_barbitur', 'rdkit_fr_benzodiazepine', 'rdkit_fr_diazo', 'rdkit_fr_dihydropyridine', 'rdkit_fr_isocyan', 'rdkit_fr_isothiocyan', 'rdkit_fr_nitroso', 'rdkit_fr_thiocyan']


In [32]:
X_train [['rdkit_NumRadicalElectrons', 'rdkit_SMR_VSA8', 'rdkit_SlogP_VSA9', 'rdkit_fr_barbitur', 'rdkit_fr_benzodiazepine', 'rdkit_fr_diazo', 'rdkit_fr_dihydropyridine', 'rdkit_fr_isocyan', 'rdkit_fr_isothiocyan', 'rdkit_fr_nitroso', 'rdkit_fr_thiocyan']].describe()

,rdkit_NumRadicalElectrons,rdkit_SMR_VSA8,rdkit_SlogP_VSA9,rdkit_fr_barbitur,rdkit_fr_benzodiazepine,rdkit_fr_diazo,rdkit_fr_dihydropyridine,rdkit_fr_isocyan,rdkit_fr_isothiocyan,rdkit_fr_nitroso,rdkit_fr_thiocyan
count,8309.0,8297.0,8323.0,8311.0,8308.0,8300.0,8303.0,8303.0,8308.0,8303.0,8287.0
mean,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
std,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
50%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
75%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
max,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [33]:
# remove constant columns from train and test
X_train = X_train.drop(columns=constant_cols)
X_test = X_test.drop(columns=constant_cols)

print(f"Final train matrix: {X_train.shape}")
print(f"Final test matrix:  {X_test.shape}")

Final train matrix: (8475, 206)
Final test matrix:  (2119, 206)


In [39]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    median_absolute_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

In [35]:
# Metric helper for MAPE percentage
def evaluate_mape(y_true, y_pred):
    return mean_absolute_percentage_error(y_true, y_pred) * 100

In [36]:
models = {
    "Decision Tree": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                DecisionTreeRegressor(
                    max_depth=10, min_samples_leaf=5, random_state=42
                ),
            ),
        ]
    ),
    "Gradient Boosting": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                GradientBoostingRegressor(
                    n_estimators=100,
                    learning_rate=0.1,
                    max_depth=5,
                    random_state=42,
                ),
            ),
        ]
    ),
    "XGBoost": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                XGBRegressor(
                    n_estimators=100,
                    learning_rate=0.05,
                    max_depth=6,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}

In [37]:
gkf = GroupKFold(n_splits=5)
groups = train_df["molecule_chembl_id"]

In [40]:
results = []

for name, pipeline in models.items():
    oof_preds = np.zeros(len(X_train))

    for fold, (train_idx, val_idx) in enumerate(
        gkf.split(X_train, y_train, groups=groups)
    ):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]

        # Fit imputer + model ONLY on training fold
        pipeline.fit(X_tr, y_tr)

        # Predict on validation fold
        oof_preds[val_idx] = pipeline.predict(X_va)

    # Compute comprehensive Out-Of-Fold (OOF) Metrics
    mape = evaluate_mape(y_train, oof_preds)
    mae = mean_absolute_error(y_train, oof_preds)
    medae = median_absolute_error(y_train, oof_preds)
    rmse = root_mean_squared_error(y_train, oof_preds)
    r2 = r2_score(y_train, oof_preds)

    results.append(
        {
            "Model": name,
            "OOF MAPE (%)": mape,
            "OOF MAE": mae,
            "OOF MedAE": medae,
            "OOF RMSE": rmse,
            "OOF R2 Score": r2,
        }
    )

# Compare cross-validation performance across models
df_results = pd.DataFrame(results).sort_values(
    by="OOF MAPE (%)", ascending=True
)
display(df_results)

,Model,OOF MAPE (%),OOF MAE,OOF MedAE,OOF RMSE,OOF R2 Score
2,XGBoost,9.010490e+15,0.616648,0.491824,0.808665,0.564921
1,Gradient Boosting,9.170212e+15,0.617888,0.494281,0.811313,0.562067
0,Decision Tree,1.154998e+16,0.722917,0.548184,0.986849,0.352063


In [43]:
train_df[target_col].describe()

count    8475.000000
mean        0.769836
std         1.226056
min        -3.107905
25%        -0.107905
50%         0.903090
75%         1.755875
max         7.806180
Name: target_log10, dtype: float64

In [46]:
train_df[train_df[target_col] == 0].shape

(279, 226)

In [47]:

# Calculate minimum shift constant to push all target values strictly > 1.0
shift_constant = abs(min(0, y_train.min())) + 1.0
y_train_shifted = y_train + shift_constant


# Safe MAPE helper for original unshifted target (filtering |y| <= 1e-3)
def evaluate_mape_safe(y_true, y_pred, eps=1e-3):
    mask = np.abs(y_true) > eps
    return (
        np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    )


# Define pipelines
models = {
    "Decision Tree": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                DecisionTreeRegressor(
                    max_depth=10, min_samples_leaf=5, random_state=42
                ),
            ),
        ]
    ),
    "Gradient Boosting": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                GradientBoostingRegressor(
                    n_estimators=100,
                    learning_rate=0.1,
                    max_depth=5,
                    random_state=42,
                ),
            ),
        ]
    ),
    "XGBoost": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                XGBRegressor(
                    n_estimators=100,
                    learning_rate=0.05,
                    max_depth=6,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}

# GroupKFold by molecule_chembl_id
gkf = GroupKFold(n_splits=5)
groups = train_df["molecule_chembl_id"]

results = []

for name, pipeline in models.items():
    oof_preds_shifted = np.zeros(len(X_train))

    for fold, (train_idx, val_idx) in enumerate(
        gkf.split(X_train, y_train_shifted, groups=groups)
    ):
        X_tr, y_tr_s = (
            X_train.iloc[train_idx],
            y_train_shifted.iloc[train_idx],
        )
        X_va = X_train.iloc[val_idx]

        # Fit on shifted target
        pipeline.fit(X_tr, y_tr_s)

        # Predict shifted values
        oof_preds_shifted[val_idx] = pipeline.predict(X_va)

    # Invert shift back to original scale
    oof_preds = oof_preds_shifted - shift_constant

    # Evaluate metrics
    mape_shifted_scale = (
        mean_absolute_percentage_error(y_train_shifted, oof_preds_shifted) * 100
    )
    mape_safe_original = evaluate_mape_safe(y_train.values, oof_preds)
    mae = mean_absolute_error(y_train, oof_preds)
    medae = median_absolute_error(y_train, oof_preds)
    rmse = root_mean_squared_error(y_train, oof_preds)
    r2 = r2_score(y_train, oof_preds)

    results.append(
        {
            "Model": name,
            "Shifted-Scale MAPE (%)": mape_shifted_scale,
            "Safe-Original MAPE (%)": mape_safe_original,
            "OOF MAE": mae,
            "OOF MedAE": medae,
            "OOF RMSE": rmse,
            "OOF R2 Score": r2,
        }
    )

# Display evaluation table
df_results = pd.DataFrame(results).sort_values(
    by="Safe-Original MAPE (%)", ascending=True
)
display(df_results)

,Model,Shifted-Scale MAPE (%),Safe-Original MAPE (%),OOF MAE,OOF MedAE,OOF RMSE,OOF R2 Score
2,XGBoost,14.483763,94.135044,0.616648,0.491826,0.808665,0.564921
1,Gradient Boosting,14.486693,97.616453,0.617665,0.492918,0.811207,0.562181
0,Decision Tree,16.753622,125.250657,0.722924,0.548094,0.988766,0.349544
